# OBM Spare Parts EDA
**Dashboard page:** Outboard Motor (OBM) Parts EDA
**Tabs:** Orders EDA (Stage 4) · Sales EDA (Stage 5)

**OBM subset:** records where `dealer_type = 'OBM'` in orders_clean / sales_clean.

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
orders = load("orders_clean.parquet")
sales  = load("sales_clean.parquet")

obm_ord = orders[orders.get("dealer_type","").eq("OBM")] if "dealer_type" in orders.columns else pd.DataFrame()
obm_sal = sales[sales.get("dealer_type","").eq("OBM")]   if "dealer_type" in sales.columns else pd.DataFrame()

print(f"OBM orders : {len(obm_ord):,}")
print(f"OBM sales  : {len(obm_sal):,}")
if len(obm_ord)==0:
    print()
    print("No OBM records found. Showing all dealer_type values:")
    if "dealer_type" in orders.columns:
        print(orders["dealer_type"].value_counts().to_string())


## Tab 1 — OBM Orders EDA

In [ ]:
if len(obm_ord)>0:
    date_col  = next((c for c in ["Created On","Document Date"] if c in obm_ord.columns),None)
    qty_col   = "Order Quantity (Item)" if "Order Quantity (Item)" in obm_ord.columns else None
    val_col   = "Net Value (Item)"      if "Net Value (Item)"      in obm_ord.columns else None
    dealer_col= next((c for c in ["Dealer Name","Sold-To Party Name"] if c in obm_ord.columns),None)

    if date_col:
        obm_ord2 = obm_ord.copy()
        obm_ord2["month"] = pd.to_datetime(obm_ord2[date_col],errors="coerce").dt.to_period("M")
        mv = obm_ord2.groupby("month").agg(qty=(qty_col,"sum") if qty_col else ("month","count"))
        mv.index = mv.index.astype(str)
        fig,ax = plt.subplots(figsize=(13,4))
        mv.iloc[:,0].plot(ax=ax,color=PALETTE[0],lw=2)
        ax.set_title("Monthly OBM Order Volume"); plt.xticks(rotation=45,ha="right")
        plt.tight_layout(); plt.show()

    if dealer_col:
        top = obm_ord[dealer_col].value_counts().head(15)
        fig,ax = plt.subplots(figsize=(11,4))
        top.sort_values().plot(kind="barh",ax=ax,color=PALETTE[2],edgecolor="white")
        ax.set_title("Top 15 OBM Dealers by Order Count"); plt.tight_layout(); plt.show()

    # Rejection log subset
    rej_mask = obm_ord["fill_rate"]==0 if "fill_rate" in obm_ord.columns else pd.Series([False]*len(obm_ord))
    print(f"Fully short-shipped OBM lines: {rej_mask.sum():,}")
    if rej_mask.sum()>0:
        print(obm_ord[rej_mask].head(10).to_string())
else:
    print("No OBM order data available.")


## Tab 2 — OBM Sales EDA

In [ ]:
if len(obm_sal)>0:
    net_col  = "Net Sales"   if "Net Sales"   in obm_sal.columns else None
    qty_col  = "SlsVolQty"   if "SlsVolQty"   in obm_sal.columns else None
    prov_col = "Province"    if "Province"    in obm_sal.columns else None
    date_col = "Billing Date" if "Billing Date" in obm_sal.columns else None

    if prov_col and net_col:
        prov = obm_sal.groupby(prov_col)[net_col].sum().sort_values(ascending=False)
        fig,ax = plt.subplots(figsize=(11,4))
        prov.plot(kind="bar",ax=ax,color=PALETTE[0],edgecolor="white")
        ax.set_title("OBM Net Sales by Province (LKR)")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e6:.1f}M"))
        plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()
        print(prov.to_string())
else:
    print("No OBM sales data available.")
